In [ ]:
# ============================================================
# ISOLATION FOREST - POLLUTION ANOMALY DETECTION
# Pune CPCB - All Stations
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import IsolationForest


# ============================================================
# 1. LOAD CLEANED HOURLY DATA
# ============================================================

file_path = "../data/processed/pune_cpcb_cleaned_hourly.csv"

if not os.path.exists(file_path):
    file_path = "data/processed/pune_cpcb_cleaned_hourly.csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)


# ============================================================
# 2. TIMESTAMP
# ============================================================

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce"
)

df = df.dropna(
    subset=["timestamp"]
)


# ============================================================
# 3. CHECK PM2.5
# ============================================================

if "pm25" not in df.columns:
    raise ValueError(
        "pm25 column not found in dataset."
    )

df["pm25"] = pd.to_numeric(
    df["pm25"],
    errors="coerce"
)


# ============================================================
# 4. SELECT POLLUTION FEATURES
# ============================================================

possible_features = [
    "pm25",
    "pm10",
    "no2",
    "so2",
    "co",
    "ozone",
    "temperature",
    "humidity",
    "wind_speed",
    "wind_direction",
    "rainfall"
]

features = [
    col for col in possible_features
    if col in df.columns
]


print("\nFeatures used:")
print(features)


# ============================================================
# 5. CONVERT FEATURES TO NUMERIC
# ============================================================

for col in features:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )


# ============================================================
# 6. HANDLE INFINITE VALUES
# ============================================================

df[features] = df[features].replace(
    [np.inf, -np.inf],
    np.nan
)


# ============================================================
# 7. REMOVE ROWS WHERE PM2.5 IS MISSING
# ============================================================

df = df.dropna(
    subset=["pm25"]
).copy()


# ============================================================
# 8. FILL OTHER MISSING VALUES
# ============================================================

# Use median values for the ML model

for col in features:

    if col != "pm25":

        df[col] = df[col].fillna(
            df[col].median()
        )


# ============================================================
# 9. PREPARE X
# ============================================================

X = df[features].copy()


# ============================================================
# 10. TRAIN ISOLATION FOREST
# ============================================================

model = IsolationForest(

    n_estimators=300,

    contamination=0.02,

    random_state=42,

    n_jobs=-1
)


print("\nTraining Isolation Forest...")

model.fit(X)

print("Training completed!")


# ============================================================
# 11. PREDICT ANOMALIES
# ============================================================

df["anomaly_prediction"] = model.predict(X)

df["anomaly_score"] = model.decision_function(X)


# Isolation Forest:
#
#  1  = normal
# -1  = anomaly


df["is_anomaly"] = (
    df["anomaly_prediction"] == -1
)


# ============================================================
# 12. ANOMALY LABEL
# ============================================================

df["anomaly_type"] = np.where(
    df["is_anomaly"],
    "Anomaly",
    "Normal"
)


# ============================================================
# 13. ANOMALY SUMMARY
# ============================================================

total_records = len(df)

total_anomalies = df[
    "is_anomaly"
].sum()

anomaly_percentage = (
    total_anomalies /
    total_records
) * 100


print("\n==========================================")
print("ISOLATION FOREST RESULTS")
print("==========================================")

print(
    "Total records:",
    total_records
)

print(
    "Anomalies:",
    total_anomalies
)

print(
    "Anomaly percentage:",
    round(
        anomaly_percentage,
        2
    ),
    "%"
)


# ============================================================
# 14. ANOMALIES BY STATION
# ============================================================

station_anomalies = (
    df.groupby("station_name")
      .agg(
          total_records=("pm25", "count"),
          anomalies=("is_anomaly", "sum"),
          avg_pm25=("pm25", "mean"),
          max_pm25=("pm25", "max")
      )
      .reset_index()
)

station_anomalies[
    "anomaly_percentage"
] = (
    station_anomalies["anomalies"]
    /
    station_anomalies["total_records"]
) * 100


station_anomalies = station_anomalies.sort_values(
    "anomaly_percentage",
    ascending=False
)


print("\n==========================================")
print("ANOMALIES BY STATION")
print("==========================================")

print(
    station_anomalies.to_string(
        index=False
    )
)


# ============================================================
# 15. TOP ANOMALY EVENTS
# ============================================================

anomalies = df[
    df["is_anomaly"]
].copy()


anomalies = anomalies.sort_values(
    "anomaly_score"
)


print("\n==========================================")
print("TOP 20 ANOMALY EVENTS")
print("==========================================")

display_columns = [
    "timestamp",
    "station_name",
    "pm25"
]

extra_columns = [
    "pm10",
    "no2",
    "so2",
    "co",
    "ozone"
]

for col in extra_columns:

    if col in df.columns:
        display_columns.append(col)

display_columns.append(
    "anomaly_score"
)

print(
    anomalies[
        display_columns
    ].head(20).to_string(
        index=False
    )
)


# ============================================================
# 16. SAVE ANOMALY DATA
# ============================================================

output_dir = "../data/processed"

if not os.path.exists(output_dir):
    output_dir = "data/processed"

os.makedirs(
    output_dir,
    exist_ok=True
)


output_file = os.path.join(
    output_dir,
    "isolation_forest_anomalies.csv"
)


df.to_csv(
    output_file,
    index=False
)


print(
    "\nFull anomaly dataset saved to:"
)

print(output_file)


# ============================================================
# 17. SAVE ONLY ANOMALIES
# ============================================================

anomaly_file = os.path.join(
    output_dir,
    "pollution_anomaly_events.csv"
)


anomalies.to_csv(
    anomaly_file,
    index=False
)


print(
    "Anomaly events saved to:"
)

print(anomaly_file)


# ============================================================
# 18. VISUALIZE PM2.5 ANOMALIES
# ============================================================

plt.figure(
    figsize=(15, 6)
)


normal = df[
    ~df["is_anomaly"]
]

abnormal = df[
    df["is_anomaly"]
]


plt.plot(
    normal["timestamp"],
    normal["pm25"],
    linewidth=0.7,
    label="Normal"
)


plt.scatter(
    abnormal["timestamp"],
    abnormal["pm25"],
    s=12,
    label="Anomaly"
)


plt.xlabel(
    "Timestamp"
)

plt.ylabel(
    "PM2.5"
)

plt.title(
    "Pune PM2.5 Anomaly Detection using Isolation Forest"
)

plt.legend()

plt.xticks(
    rotation=45
)

plt.tight_layout()

plt.show()


# ============================================================
# 19. STATION ANOMALY BAR CHART
# ============================================================

plt.figure(
    figsize=(12, 6)
)


plt.bar(
    station_anomalies["station_name"],
    station_anomalies["anomaly_percentage"]
)


plt.xlabel(
    "Station"
)

plt.ylabel(
    "Anomaly Percentage (%)"
)

plt.title(
    "Pollution Anomalies by Pune Station"
)

plt.xticks(
    rotation=60,
    ha="right"
)

plt.tight_layout()

plt.show()


# ============================================================
# 20. PM2.5 DISTRIBUTION
# ============================================================

plt.figure(
    figsize=(10, 6)
)


plt.hist(
    normal["pm25"],
    bins=50,
    alpha=0.7,
    label="Normal"
)

plt.hist(
    abnormal["pm25"],
    bins=30,
    alpha=0.7,
    label="Anomaly"
)


plt.xlabel(
    "PM2.5"
)

plt.ylabel(
    "Frequency"
)

plt.title(
    "Normal vs Anomalous PM2.5"
)

plt.legend()

plt.tight_layout()

plt.show()


# ============================================================
# 21. MOST ANOMALOUS STATION
# ============================================================

top_station = station_anomalies.iloc[0]


print("\n==========================================")
print("MOST ANOMALOUS STATION")
print("==========================================")

print(
    "Station:",
    top_station["station_name"]
)

print(
    "Anomalies:",
    int(top_station["anomalies"])
)

print(
    "Anomaly percentage:",
    round(
        top_station["anomaly_percentage"],
        2
    ),
    "%"
)

print(
    "Average PM2.5:",
    round(
        top_station["avg_pm25"],
        2
    )
)

print(
    "Maximum PM2.5:",
    round(
        top_station["max_pm25"],
        2
    )
)


print("\n==========================================")
print("ISOLATION FOREST COMPLETED")
print("==========================================")

